These solutions pertain to the CTF challenges available on <a href="https://pwnable.kr/play.php">pwnable.kr</a>!

<ol>
    <li> <a href="#toddlers-bottle">Toddler's Bottle</a> </li>
    <ul>
        <li><a href="#fd">fd</a> </li>
        <li><a href="#collision">collision</a> </li>
    </ul>
    <li> <a href="#rookiss">Rookiss</a> </li>
    <li> <a href="#grotesque">Grotesque</a> </li>
    <li> <a href="#hackerssecret">Hacker's Secret</a> </li>
</ol>

## <a name="toddlers-bottle"></a>Toddler's Bottle


#### <a name="fd"></a><font color="red">fd</font>
We first run the ls command to see what we are working with. We have three files:

* fd
* fd.c
* flag

As expected, the `flag` file does not have read permissions for us. So, let's examine the code. We can run the following command to output the contents of the `fd.c` file:

```shell
cat fd.c
```

Upon examining the contents of the program, it is clear that we need a way to fill the string "LETMEWIN\n" into the `buf`. How? When we examine the read syscall closely, we see that it reads in from the file descriptor stored in `int fd`, which is essentially argv[1] converted to an int minus `0x1234`. Now, we need to understand the following building-blocks:

* STDIN is 0: If we manage to set `int fd` to 0, it will allow us to write directly from the terminal. This means whatever we type into the terminal will be entered into `buf`, which is great because we can type "LETMEWIN\n".

However, we see that:

```C
int fd = atoi(argv[1]) - 0x1234;
```

To see what value `0x1234` (hex) is in decimal, we can simply type this into the shell:

```shell
printf "%d\n" 0x1234
```

Output: `4660`

So, to set `fd=0`, we have to pass in `4660` as the input. We run the program:

```shell
./fd 4660
```

and voila, we are allowed to enter a string. Now we can simply write "LETMEWIN\n" (without the quotes, and \n meaning an actual press of the return key), and behold the flag is:

`flag`: mommy! I think I know what a file descriptor is!!

<b>Question:</b> How are we not able to use `cat flag` to output the contents due to permission issues, but when we ourselves run this `fd` program with the correct input, we are able to access the contents of the file? After all, the process `fd` was run by us non-privileged users.

<b>Answer:</b> If you run `ls -l` and look at the permissions, you will notice an `s` in the owner's execute permissions. Something like: `-r-sr-x---`. This is called the `setuid` bit, and any program with the `setuid` bit set will run with the permissions of the owner. Therefore, when we correctly execute the program, it is able to print the content of the file "flag" as it is running with the permissions of the owner of the "flag" file, rather than with our permissions.

#### <a name="collision"></a><font color="red">collision</font>